<a href="https://colab.research.google.com/github/RayanMohammed/de-pipeline/blob/main/FHIR_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip /content/drive/MyDrive/data/FHIR_data/data.zip -d /content/patients_data

Archive:  /content/drive/MyDrive/data/FHIR_data/data.zip
  inflating: /content/patients_data/Sharolyn456_Huels583_5b24c87b-6223-f5b4-51e9-82051159bd1d.json  
  inflating: /content/patients_data/Warner493_McKenzie376_13472219-c176-990a-641f-14cf9d4d8480.json  
  inflating: /content/patients_data/Terry864_Glover433_d20a36fc-23ba-8462-bf39-864000fbf25f.json  
  inflating: /content/patients_data/Georgiann138_Dickinson688_53e4891a-9108-67ef-d973-3b6a98404249.json  
  inflating: /content/patients_data/Dominic463_Parker433_03a7cc66-36c5-356c-419f-93bfbd7b558d.json  
  inflating: /content/patients_data/Gertrudis163_Tasia358_Paucek755_a8286256-4ef3-c614-4371-085d7c8cfcb2.json  
  inflating: /content/patients_data/Latrina689_Hilpert278_34b8aa06-9904-df2f-e571-c6c5e767f24b.json  
  inflating: /content/patients_data/Tuan998_Bernhard322_68988c92-ff15-280a-9b96-5d8f72e7b0ef.json  
  inflating: /content/patients_data/Harvey63_Ernser583_3ab40d52-44be-ae07-2177-2875cc65f5a8.json  
  inflating: /content

In [3]:
!pip install supabase duckdb pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.8 MB/s eta 0:00:00


In [7]:
import duckdb
import pandas as pd
import glob, json
from google.colab import userdata
from supabase import create_client


url = userdata.get("SUPABASE_URL")
key = userdata.get("SUPABASE_KEY")
supabase = create_client(url, key)

full_list = glob.glob('/content/patients_data/*.json')
valid_files = []

for item in full_list:
    try:
        with open(item, 'r') as file:
            new_dict = json.load(file)
        if new_dict.get('resourceType') == 'Bundle' and 'entry' in new_dict:
            first_resource = new_dict['entry'][0]['resource']
            if first_resource.get('resourceType') == 'Patient':
                valid_files.append(item)
    except Exception as e:
        pass


query = f"""
SELECT
    flattened_entry.resource.id AS id,
    flattened_entry.resource.gender AS gender,
    flattened_entry.resource.birthDate AS birth_date
FROM (
    SELECT
        UNNEST(entry) as flattened_entry
    FROM read_json_auto({valid_files}, maximum_object_size=104857600, union_by_name=true)
)
WHERE flattened_entry.resource.resourceType = 'Patient'
"""

conn = duckdb.connect()
df = conn.execute(query).df()

df['id'] = df['id'].astype(str)
if 'birth_date' in df.columns:
    df['birth_date'] = df['birth_date'].astype(str)

records = df.to_dict(orient="records")

supabase.table('patients').insert(records).execute()
print(f"inserted {len(records)} records into supabase")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

inserted 109 records into supabase
